# n-gram 언어 모델링

In [2]:
import nltk
from nltk.util import ngrams
from collections import Counter



In [3]:
text = '오늘은 날씨가 맑다. 오늘은 기분이 좋다. 오늘은 사람이 많다. 오늘은 수업이 좋다.'

In [5]:
tokens = nltk.word_tokenize(text)

tokens

['오늘은',
 '날씨가',
 '맑다',
 '.',
 '오늘은',
 '기분이',
 '좋다',
 '.',
 '오늘은',
 '사람이',
 '많다',
 '.',
 '오늘은',
 '수업이',
 '좋다',
 '.']

In [6]:
unigram = tokens
bigram = list(ngrams(tokens, 2))

unigram_freq = Counter(unigram)
bigram_freq = Counter(bigram)

print(unigram_freq)
print(bigram_freq)

Counter({'오늘은': 4, '.': 4, '좋다': 2, '날씨가': 1, '맑다': 1, '기분이': 1, '사람이': 1, '많다': 1, '수업이': 1})
Counter({('.', '오늘은'): 3, ('좋다', '.'): 2, ('오늘은', '날씨가'): 1, ('날씨가', '맑다'): 1, ('맑다', '.'): 1, ('오늘은', '기분이'): 1, ('기분이', '좋다'): 1, ('오늘은', '사람이'): 1, ('사람이', '많다'): 1, ('많다', '.'): 1, ('오늘은', '수업이'): 1, ('수업이', '좋다'): 1})


In [16]:
for (w1, w2), freq in bigram_freq.items():
    prob = freq / unigram_freq[w1]
    print(f'P({w2}|{w1}) = {prob:.3f}')

P(날씨가|오늘은) = 0.250
P(맑다|날씨가) = 1.000
P(.|맑다) = 1.000
P(오늘은|.) = 0.750
P(기분이|오늘은) = 0.250
P(좋다|기분이) = 1.000
P(.|좋다) = 1.000
P(사람이|오늘은) = 0.250
P(많다|사람이) = 1.000
P(.|많다) = 1.000
P(수업이|오늘은) = 0.250
P(좋다|수업이) = 1.000


In [25]:
# perplexity 수동 구현(모델이 테스트 데이터에서 얼마나 적은 불확실성을 가지며 다음 단어 예측하는지)
import math 

def compute_bigram_perplexity(test_text, unigram_freq, bigram_freq):
    test_tokens = nltk.word_tokenize(test_text)     # 테스트 문장 토큰화
    test_bigrams = list(ngrams(test_tokens, 2))     # 바이그램 생성
    
    log_prob_sum = 0        # 로그확률 합
    N = len(test_bigrams)   # 평균용 바이그램 갯수
    
    for bigram in test_bigrams:
        w1, w2 = bigram
        # P(w2|w1) 계산 (get(key, n) : key 있으면 key , key 없으면 n)
        prob = bigram_freq.get(bigram, 0) /unigram_freq.get(w1, 1)  
        if prob == 0 :
            prob = 1e-10    # prob이 0이면 매우 미세한 값으로 대체
        log_prob_sum += math.log2(prob) # log 확률 누적
        
    cross_entropy = -log_prob_sum/ N        # Cross-Entropy =  -(1/N) * / ∑ log2 P
    
    perplexity = math.pow(2, cross_entropy) # Perplexity = 2^(Cross-Entropy)
    
    return perplexity
    

In [26]:
train_text = '자연어 처리는 재미있다. 자연어 처리는 어렵지만 도전하고 싶다. 오늘은 날씨가 좋다.'

train_tokens = nltk.word_tokenize(train_text)

unigram = train_tokens
bigrams = list(ngrams(train_tokens, 2))

unigram_freq = Counter(unigram)
bigram_freq = Counter(bigrams)

In [27]:
test_sentences = [
    '자연어 처리는 재미있다.',
    '자연어 처리는 어렵지만 도전하고 싶다.',
    '오늘은 날씨가 좋다.',
    '오늘은 날씨가 맑다.',
    '오늘은 기분이 좋다.',
    '오늘은 사람이 많다.',
    '오늘은 수업이 좋다.'
]

for sentence in test_sentences:
    pp = compute_bigram_perplexity(sentence, unigram_freq, bigram_freq)
    print(f"{sentence}의 Perplexity :{pp}")

자연어 처리는 재미있다.의 Perplexity :1.2599210498948732
자연어 처리는 어렵지만 도전하고 싶다.의 Perplexity :1.148698354997035
오늘은 날씨가 좋다.의 Perplexity :1.0
오늘은 날씨가 맑다.의 Perplexity :4641588.833612777
오늘은 기분이 좋다.의 Perplexity :4641588.833612777
오늘은 사람이 많다.의 Perplexity :10000000000.000008
오늘은 수업이 좋다.의 Perplexity :4641588.833612777


학습시에 나왔던 지표로 사용하는 문장이 많은 텍스트는 perplexity가 낮게 형성되고,  
학습에 없던 조합이 많으면 높게 형성된다.

In [ ]:
from nltk.lm import  MLE        # MLE 기반 n-gram 언어 모델
from nltk.lm.preprocessing import padded_everygram_pipeline # n-gram 언어모델 입력데이터 패딩
from nltk.util import bigrams   # 바이그램 생성

train_tokens = [['I', "love", "NLP"], ['I', "love", "Python"]]

# 바이그램 학습 데이터 생성 : 문장 시작/ 끝 토큰 추가 + unigram 과 bigram 생성
train_data, vocab = padded_everygram_pipeline(2, train_tokens)  # 2: 바이그램 생성

model = MLE(2)  # MLE 방식의 바이그램 언어모델
model.fit(train_data, vocab)    # 학습데이터와 단어사전 전달

test_tokens = nltk.word_tokenize('I love Python')
test_bigrams = list(bigrams(test_tokens))

perplexity = model.perplexity(test_bigrams)
perplexity

1.4142135623730951